# Just thought of sharing this, SIRS/MAMs

# DO THIS WHEN U HAVE 8GB RAM. 

# Just showing the reasoning ability through a particular MODEL WHICH OLLAMA SUPPORTS
# Also have given snippets of reasoning through OPENAI, GEMINI and CLAUDE

# Notebook 03: Reasoning and Generation Controls

## Learning objectives

By the end of this notebook, you will be able to:

1. Distinguish prompting from model configuration.
2. Discover a model's supported thinking controls.
3. Compare answers under different thinking settings.
4. Explore temperature and generation limits separately.
5. Recognize how reasoning controls differ across providers.

**Keywords:** Thinking • Effort • Temperature • Tokens • Evaluation

## Main principle

Change one parameter at a time.

Evaluate correctness and usefulness alongside execution time and token use.

## 1. Understand the controls

| Control | What it affects | What it does not guarantee |
|---|---|---|
| Prompt instructions | Task, evidence, constraints, and presentation | Correctness |
| Thinking / reasoning effort | Model effort on supported models | A correct answer |
| Temperature | Sampling variability | Creativity or factual accuracy |
| Generation limit | Maximum generated tokens | A complete answer |
| Requested answer length | Desired visible response length | Exact compliance |

### Important distinctions

- A long answer does not prove that the model reasoned better.
- A short answer can follow substantial internal reasoning.
- Temperature is not a reasoning-depth control.
- A generation limit is not the same as a reasoning budget.
- Writing "reasoning effort: high" inside a prompt does not set an API parameter.

In [ ]:
%pip install requests pandas

In [ ]:
!ollama pull qwen3:8b

In [11]:
import json
import time
import random
import requests
import pandas as pd

from IPython.display import display, Markdown

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3:8b"  # SELECT MRODELS THAT HAVE REASONING MODE

REQUEST_TIMEOUT = 600
DEFAULT_TOKEN_LIMIT = 4096

try:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )
    response.raise_for_status()

except requests.RequestException as error:
    raise RuntimeError(
        "Cannot reach Ollama. Start Ollama and check OLLAMA_URL."
    ) from error

available_models = [
    item["name"]
    for item in response.json().get("models", [])
]

print("Installed models:", available_models)

if MODEL not in available_models:
    raise RuntimeError(
        f"{MODEL!r} is not installed. "
        f"Run 'ollama pull {MODEL}' in a terminal "
        "or choose an installed model."
    )

Installed models: ['nomic-embed-text:latest', 'llama3.2:1b', 'qwen2.5:1.5b']


RuntimeError: 'qwen3:8b' is not installed. Run 'ollama pull qwen3:8b' in a terminal or choose an installed model.

## 2. Discover supported thinking controls

Thinking controls depend on the model and Ollama version.

Examples of possible supported values:

- false and true: on/off controls.
- "low", "medium", and "high": named levels.
- Only false: no supported thinking mode.

Named levels are not universally available.

If thinking metadata is absent, do not conclude that the model cannot reason.
Check the model documentation and Ollama version.

In [5]:
response = requests.post(
    f"{OLLAMA_URL}/api/show",
    json={"model": MODEL},
    timeout=30
)
response.raise_for_status()

model_info = response.json()
thinking_info = model_info.get("thinking")

print("Thinking metadata:")
print(json.dumps(thinking_info, indent=2))

SUPPORTED_THINKING_VALUES = (
    thinking_info.get("values", [])
    if isinstance(thinking_info, dict)
    else []
)

print("\nSupported values:", SUPPORTED_THINKING_VALUES)

# Only use an override after checking the documentation for your exact model.
# Example for a documented on/off model: [False, True]
DOCUMENTED_OVERRIDE = []

THINKING_VALUES = (
    SUPPORTED_THINKING_VALUES
    if SUPPORTED_THINKING_VALUES
    else DOCUMENTED_OVERRIDE
)

if not THINKING_VALUES:
    print(
        "\nNo thinking settings were discovered. "
        "Check model documentation or update Ollama before "
        "running the thinking comparison."
    )

Thinking metadata:
null

Supported values: []

No thinking settings were discovered. Check model documentation or update Ollama before running the thinking comparison.


## 3. A configurable model-call function

Notice where the parameters appear:

- think: top-level request field.
- temperature: inside options.
- num_predict: inside options.

Setting think=None in our Python function omits the field,
so Ollama uses the model's default behavior.

The generated-token count reported by Ollama is not a separate
reasoning-token count.

In [6]:
def call_ollama(
    prompt,
    think=None,
    temperature=0.2,
    token_limit=DEFAULT_TOKEN_LIMIT,
    json_output=False
):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": token_limit
        }
    }

    if think is not None:
        payload["think"] = think

    if json_output:
        payload["format"] = "json"

    start = time.perf_counter()

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()

    except requests.RequestException as error:
        raise RuntimeError(
            f"Ollama request failed: {error}"
        ) from error

    data = response.json()
    message = data.get("message", {})

    load_ns = data.get("load_duration")

    return {
        "answer": message.get("content", ""),
        "thinking": message.get("thinking", ""),
        "seconds": round(time.perf_counter() - start, 2),
        "load_seconds": (
            round(load_ns / 1e9, 3)
            if isinstance(load_ns, (int, float))
            else None
        ),
        "generated_tokens": data.get("eval_count"),
        "stop_reason": data.get("done_reason"),
        "think_setting": think,
        "temperature": temperature,
        "token_limit": token_limit
    }


def show_result(result):
    display(Markdown(result["answer"] or "*No final answer returned.*"))

    print(
        f"Time: {result['seconds']} seconds | "
        f"Generated tokens: {result['generated_tokens']} | "
        f"Stop reason: {result['stop_reason']}"
    )

## 4. Use tasks with checkable answers

We will use two project-scheduling problems.

### Task A: Sequential tasks

A takes 2 days.
B takes 3 days and starts after A finishes.
C takes 1 day and starts after B finishes.

Total duration = 2 + 3 + 1 = 6 days.

A 5-day deadline cannot be met.

### Task B: Parallel branches

A takes 2 days.
After A, B takes 4 days and C takes 3 days.
B and C can run in parallel.
D takes 2 days and starts after both B and C finish.

Total duration = 2 + max(4, 3) + 2 = 8 days.

A 7-day deadline cannot be met.

Assume adequate resources, no gaps, and zero task-transition time.

In [7]:
TASKS = [
    {
        "id": "Sequential",
        "description": """
A takes 2 days.
B takes 3 days and starts after A finishes.
C takes 1 day and starts after B finishes.
The deadline is 5 days.
""".strip(),
        "expected_days": 6,
        "expected_deadline": False
    },
    {
        "id": "Parallel",
        "description": """
A takes 2 days.
After A finishes, B takes 4 days and C takes 3 days.
B and C can run in parallel.
D takes 2 days and starts after both B and C finish.
The deadline is 7 days.
""".strip(),
        "expected_days": 8,
        "expected_deadline": False
    }
]


def scheduling_prompt(task):
    return f"""
Calculate the minimum project duration.

Assume:
- Adequate resources.
- No gaps between tasks.
- Zero task-transition time.
- Tasks cannot be shortened.

Task:
{task["description"]}

Return only a JSON object with these fields:
{{
  "duration_days": 0,
  "deadline_met": false,
  "calculation": "A brief calculation supporting the answer"
}}

Use a number for duration_days and a Boolean for deadline_met.
""".strip()

## 5. Compare thinking settings

Keep the prompt, temperature, and generation limit fixed.

Change only the supported thinking setting.

We use JSON output to make the final answers easier to evaluate.
JSON formatting does not guarantee correct values.

The first call may include model-loading time.
We perform a warm-up call before the comparison.

Start with one repeat for a short demonstration.
Increase repeats to explore variability.

In [8]:
REPEATS = 1

experiment_rows = []
experiment_outputs = {}

# Use at most three supported settings for the initial demonstration.
SETTINGS_TO_TEST = THINKING_VALUES[:3]

if len(SETTINGS_TO_TEST) < 2:
    print(
        "At least two documented thinking settings are needed "
        "for this comparison. Select a compatible model."
    )

else:
    print("Warming up the model...")

    warm_up = call_ollama(
        "Reply with OK.",
        think=SETTINGS_TO_TEST[0]
    )

    jobs = [
        (task, setting, repeat)
        for task in TASKS
        for setting in SETTINGS_TO_TEST
        for repeat in range(1, REPEATS + 1)
    ]

    # Shuffle execution order to reduce a fixed ordering effect.
    random.Random(42).shuffle(jobs)

    for task, setting, repeat in jobs:
        label = json.dumps(setting)

        print(
            f"Running {task['id']} | "
            f"think={label} | repeat={repeat}",
            flush=True
        )

        result = call_ollama(
            scheduling_prompt(task),
            think=setting,
            temperature=0.2,
            json_output=True
        )

        key = (task["id"], label, repeat)
        experiment_outputs[key] = result

        parsed = None
        parse_ok = False
        schema_ok = False
        correct = False

        try:
            parsed = json.loads(result["answer"])
            parse_ok = True

            schema_ok = (
                isinstance(parsed, dict)
                and set(parsed) == {
                    "duration_days",
                    "deadline_met",
                    "calculation"
                }
                and type(parsed["duration_days"]) in (int, float)
                and type(parsed["deadline_met"]) is bool
                and isinstance(parsed["calculation"], str)
            )

            correct = (
                schema_ok
                and parsed["duration_days"] == task["expected_days"]
                and parsed["deadline_met"] is task["expected_deadline"]
                and result["stop_reason"] != "length"
            )

        except (json.JSONDecodeError, TypeError):
            pass

        experiment_rows.append({
            "Task": task["id"],
            "Thinking setting": label,
            "Repeat": repeat,
            "JSON parsed": parse_ok,
            "Schema valid": schema_ok,
            "Correct final values": correct,
            "Seconds": result["seconds"],
            "Model load seconds": result["load_seconds"],
            "Generated tokens": result["generated_tokens"],
            "Stop reason": result["stop_reason"]
        })

    experiment_df = pd.DataFrame(experiment_rows)
    display(experiment_df)

At least two documented thinking settings are needed for this comparison. Select a compatible model.


In [9]:
if experiment_rows:
    summary = (
        pd.DataFrame(experiment_rows)
        .groupby("Thinking setting", as_index=False)
        .agg(
            Accuracy=("Correct final values", "mean"),
            Mean_seconds=("Seconds", "mean"),
            Mean_generated_tokens=("Generated tokens", "mean")
        )
    )

    display(summary)

    print(
        "This is a small classroom experiment, "
        "not a general benchmark of model capability."
    )
else:
    print("Run the thinking comparison with a compatible model first.")

Run the thinking comparison with a compatible model first.


## 6. Inspect answers

Automatic checks above evaluate the duration and deadline decision.

They do not establish that the explanation is correct.

Review the brief calculation separately:

- Did the model add sequential task durations?
- Did it use the maximum duration for parallel branches?
- Did it follow the assumptions?

A correct final answer can still have an incorrect explanation.

In [10]:
for key, result in experiment_outputs.items():
    task_id, setting, repeat = key

    print(f"\n{task_id} | think={setting} | repeat={repeat}")
    print(result["answer"])

## 7. Thinking text and final answers are different fields

Some Ollama models return:

- message.thinking
- message.content

Thinking text may be absent for some settings or models.

A displayed reasoning trace should not be treated as guaranteed evidence
of how the answer was produced or whether the answer is correct.

Evaluate the final answer using facts, calculations, and external checks.

In [ ]:
# Inspect whether thinking text was returned, without printing a long trace.

for key, result in experiment_outputs.items():
    print(
        key,
        "| Thinking text returned:",
        bool(result["thinking"].strip()),
        "| Thinking characters:",
        len(result["thinking"])
    )

## 8. Temperature experiment

Temperature affects sampling variability.

We will hold the thinking setting fixed and change temperature.

The task is idea generation, so there is no single correct answer.

Compare:

- Variety.
- Relevance.
- Repetition.
- Compliance with the requested format.

Higher temperature does not guarantee more useful ideas.
Temperature 0 does not guarantee identical results on every system.

In [ ]:
# Prefer thinking off when explicitly supported.
# Otherwise use one discovered setting, or the model default.

FIXED_THINK = (
    False
    if False in THINKING_VALUES
    else THINKING_VALUES[0] if THINKING_VALUES else None
)

IDEA_PROMPT = """
Suggest three practical ways to help employees remember
a new internal reporting procedure.

Use one short sentence per idea.
Focus on low-cost actions.
Avoid repeating the same idea in different words.
""".strip()

temperature_results = []

for temperature in [0.0, 0.8]:
    for repeat in range(1, 3):
        print(
            f"\nTemperature={temperature} | Repeat={repeat}",
            flush=True
        )

        result = call_ollama(
            IDEA_PROMPT,
            think=FIXED_THINK,
            temperature=temperature
        )

        temperature_results.append(result)
        show_result(result)

## 9. Generation-limit experiment

A token limit is a hard generation ceiling.

It is not an instruction to produce a good summary.

With reasoning-capable models, a small allowance may be consumed
before a final answer is produced.

We will compare a small and a larger allowance while holding
the prompt and thinking setting fixed.

Use the stop reason to identify truncation.

In [ ]:
LIMIT_PROMPT = """
Prepare a training plan for a two-hour introductory AI workshop.

Include:
1. Learning objectives.
2. A timed agenda.
3. Two practical exercises.
4. A method to assess participant understanding.

Keep the visible answer within 200 words.
""".strip()

for limit in [64, 4096]:
    print(f"\nGeneration limit: {limit}", flush=True)

    result = call_ollama(
        LIMIT_PROMPT,
        think=FIXED_THINK,
        temperature=0.2,
        token_limit=limit
    )

    show_result(result)

    if result["stop_reason"] == "length":
        print("The generation limit was reached.")

## 10. Separate answer length from reasoning effort

These are different requirements:

- "Use more reasoning effort" changes supported model configuration.
- "Answer within 50 words" requests a short visible answer.

A model can perform substantial reasoning and return a concise result.

For difficult tasks, request a useful calculation or justification,
rather than assuming that a longer answer is better.

In [ ]:
# Choose a discovered setting; use the model default if none was discovered.
SELECTED_THINK = (
    THINKING_VALUES[-1]
    if THINKING_VALUES
    else None
)

concise_prompt = """
A takes 2 days.
After A, B takes 4 days and C takes 3 days in parallel.
D takes 2 days after both B and C finish.

What is the minimum project duration?

Answer within 50 words.
Include the calculation.
""".strip()

concise_result = call_ollama(
    concise_prompt,
    think=SELECTED_THINK,
    temperature=0.2
)

show_result(concise_result)

## 11. Reasoning controls differ across providers

| Provider / API | Main control shown here |
|---|---|
| Ollama | Top-level think field |
| OpenAI Responses | reasoning={"effort": "..."} |
| Claude Messages | output_config={"effort": "..."} |
| Gemini Interactions | generation_config={"thinking_level": "..."} |

Supported values depend on the selected model.

A provider's "high" setting is not a standardized amount of reasoning
that can be compared directly with another provider's "high".

Parameter names can also differ between APIs from the same provider.

## 12. Cloud API demonstrations

Skip this section if you are using only Ollama.

These examples require:

- Internet access.
- A provider API key.
- Access to the selected model.
- The corresponding Python SDK.

API calls can incur charges.

Examples are disabled initially.
Enable only the provider you intend to demonstrate.

The model names below are examples from current official documentation.
Use a model available to your account and check its supported settings.

In [ ]:
import os
from getpass import getpass

RUN_OPENAI = False
RUN_CLAUDE = False
RUN_GEMINI = False


def obtain_key(environment_name):
    """Use an existing environment variable or request hidden input."""
    key = os.environ.get(environment_name)

    if not key:
        key = getpass(f"Enter {environment_name}: ").strip()

    if not key:
        raise ValueError("No API key supplied.")

    return key


CLOUD_PROMPT = """
A takes 2 days.
After A, B takes 4 days and C takes 3 days in parallel.
D takes 2 days after both B and C finish.

Assume adequate resources and no gaps.
Calculate the minimum project duration.
Give the calculation and conclusion within 60 words.
""".strip()

### OpenAI: reasoning effort

The Responses API uses:

reasoning={"effort": "low"}

Supported effort values vary by model.
Do not assume that every model supports "none" or "minimal".

The example does not set temperature because sampling-parameter support
also depends on the model and configuration.

In [ ]:
if RUN_OPENAI:
    from openai import OpenAI

    client = OpenAI(
        api_key=obtain_key("OPENAI_API_KEY")
    )

    OPENAI_MODEL = "gpt-6-astra"

    for effort in ["low", "medium"]:
        start = time.perf_counter()

        response = client.responses.create(
            model=OPENAI_MODEL,
            reasoning={"effort": effort},
            input=CLOUD_PROMPT,
            max_output_tokens=4096
        )

        print(f"\nOPENAI | effort={effort}")
        print(response.output_text)
        print("Status:", response.status)
        print("Time:", round(time.perf_counter() - start, 2), "seconds")

        if response.usage:
            print("Input tokens:", response.usage.input_tokens)
            print("Output tokens:", response.usage.output_tokens)

            details = response.usage.output_tokens_details
            print(
                "Reasoning tokens:",
                getattr(details, "reasoning_tokens", None)
            )

        if response.status == "incomplete":
            print("Incomplete details:", response.incomplete_details)

else:
    print("OpenAI demonstration skipped.")

### Claude: effort

The Messages API uses:

output_config={"effort": "medium"}

Effort is guidance about overall model work, not a strict thinking-token budget.

For the Claude Opus 5.5 example below, adaptive thinking is always on.
Use effort to adjust model work.

Older Claude models can have different thinking configurations,
including explicit thinking budgets.
Do not copy thinking settings between models without checking support.

In [ ]:
if RUN_CLAUDE:
    import anthropic

    client = anthropic.Anthropic(
        api_key=obtain_key("ANTHROPIC_API_KEY")
    )

    CLAUDE_MODEL = "claude-opus-5-5"

    for effort in ["low", "medium"]:
        start = time.perf_counter()

        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=4096,
            output_config={"effort": effort},
            messages=[
                {"role": "user", "content": CLOUD_PROMPT}
            ]
        )

        answer = "\n".join(
            block.text
            for block in response.content
            if block.type == "text"
        )

        print(f"\nCLAUDE | effort={effort}")
        print(answer)
        print("Stop reason:", response.stop_reason)
        print("Time:", round(time.perf_counter() - start, 2), "seconds")
        print("Input tokens:", response.usage.input_tokens)
        print("Output tokens:", response.usage.output_tokens)

else:
    print("Claude demonstration skipped.")

### Gemini: thinking level

This example uses the Gemini Interactions API:

generation_config={"thinking_level": "low"}

For the example model, supported levels include low, medium, and high.

Gemini's Generate Content API uses a different configuration structure.
Earlier Generate Content examples may use thinking budgets for Gemini 2.5.

Always identify both the model and the API before copying parameters.

In [ ]:
if RUN_GEMINI:
    from google import genai

    client = genai.Client(
        api_key=obtain_key("GEMINI_API_KEY")
    )

    GEMINI_MODEL = "gemini-3.8-flash"

    for level in ["low", "medium"]:
        start = time.perf_counter()

        interaction = client.interactions.create(
            model=GEMINI_MODEL,
            input=CLOUD_PROMPT,
            generation_config={
                "thinking_level": level,
                "max_output_tokens": 4096
            }
        )

        print(f"\nGEMINI | thinking_level={level}")
        print(interaction.output_text)
        print(
            "Status:",
            getattr(interaction, "status", None)
        )
        print("Time:", round(time.perf_counter() - start, 2), "seconds")

else:
    print("Gemini demonstration skipped.")

## 13. Participant exercise

Choose one task with a checkable answer.

Examples:

- Calculate a project schedule.
- Diagnose a short code error.
- Check whether a proposed plan satisfies stated constraints.
- Compare options using supplied numerical criteria.

Steps:

1. Write the task and expected answer.
2. Choose two supported thinking settings.
3. Keep the prompt, temperature, and generation limit fixed.
4. Run each setting more than once.
5. Evaluate correctness and instruction compliance.
6. Compare time and generated-token use.

### Record your results

| Setting | Correct? | Requirements met? | Time | Token use |
|---|---|---|---|---|

### Discuss

- Did the higher setting improve the answer?
- Did it add time or token use?
- Were both settings already sufficient?
- Was any response truncated?
- What additional tasks would you test before choosing a default?

## Key takeaways

- Prompt instructions and API parameters serve different purposes.
- Discover supported controls before using them.
- Compare settings on tasks with clear evaluation criteria.
- Temperature is separate from reasoning effort.
- Small generation limits can prevent complete answers.
- Higher effort should earn its additional time and token use.
- Provider effort labels are not directly equivalent.
 